In [1]:
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)


Running with 7 CPU cores | 15.8GB total RAM (8.7GB available) | pyarrow 24.0.0


Variables

In [2]:
%run 0_variables.ipynb

Feature selection

In [3]:
import time as _time

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_fdu_path = resolve(f"3_Features_select/Selected_features/{_target}_feature_data_unique_{_stem}.parquet")
print(f"Loading unique feature_data parquet: {_fdu_path}", flush=True)
_t = _time.perf_counter()
feature_data_unique = pd.read_parquet(_fdu_path)
print(f"  loaded feature_data_unique: shape={feature_data_unique.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

print(f"Loading features parquet: {os.environ['FEATURES_PATH']}", flush=True)
_t = _time.perf_counter()
features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
    ('SETTLEMENTDATE', '>=', pd.Timestamp(os.environ["FEATURE_DATASET_START"])),
    ('SETTLEMENTDATE', '<=', pd.Timestamp(os.environ["FEATIRE_DATASET_END"])),
])
features = features.drop(columns=[c for c in features.columns if c in set(os.environ["TARGET_COLS"].split(","))])
features = features.loc[:pd.Timestamp(os.environ["FEATIRE_DATASET_END"])]
print(f"  loaded features: shape={features.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_tagg_path = resolve(f"3_Features_select/Selected_features/{_target}_targets_agg_{_stem}.parquet")
print(f"Loading aggregated targets parquet: {_tagg_path}", flush=True)
_t = _time.perf_counter()
future_prediction_targets_agg = pd.read_parquet(_tagg_path)
print(f"  loaded targets_agg: shape={future_prediction_targets_agg.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

def feature_data_selection(
    feature_data,
    features,
    future_prediction_targets_agg,
    subsample_start: pd.Timestamp,
    subsample_end: pd.Timestamp,
    subsample_amount: int,
):
    # Number of equally-spaced k values to evaluate between K_MIN and K_MAX — tune this
    N_K_VALUES = 10
    # Optional bounds for the k sweep (inclusive). None = use 1 / n_available_features respectively.
    K_MIN = None
    K_MAX = None
    # Number of equally-spaced horizons to evaluate — e.g. 10 tests every ~10th horizon; None = all
    N_HORIZON_STEPS = None
    N_CV_SPLITS = 3   # time-series walk-forward folds
    # Outer thread count — each task sweeps ALL folds × k for one horizon (per-horizon task layout).
    # Real values set below once n_tasks is known; placeholder here so _LGBM_CV_PARAMS can reference it.
    N_PARALLEL_JOBS = _NCPU
    _LGBM_INNER_THREADS = 1

    # Lightweight LightGBM for CV speed — not production params.
    # Tuned aggressively for ranking-of-k (not absolute MAE accuracy):
    #   - num_leaves=15: ~40% faster fits, rarely changes which k wins.
    #   - learning_rate=0.1 + n_estimators=100 (fixed): no eval_set / no early stopping (#10) avoids
    #     per-iteration validation cost (~15-25% faster). Fixed 100 trees converges fine at lr=0.1.
    #   - bagging_fraction=0.5, bagging_freq=1: ~2× faster per tree iteration; adds noise but k-ranking
    #     is robust to it.
    #   - force_col_wise=True: skips LightGBM's layout auto-detection, goes straight to column-wise
    #     (matches our Fortran-ordered X_arr, see below).
    _LGBM_CV_PARAMS = {
        "objective": "regression_l1",
        "metric": "mae",
        "n_estimators": 100,            # fixed; no early stopping
        "learning_rate": 0.1,
        "num_leaves": 15,
        "min_child_samples": 20,
        "bagging_fraction": 0.5,
        "bagging_freq": 1,
        "force_col_wise": True,
        "random_state": 42,
        "n_jobs": _LGBM_INNER_THREADS,      # sklearn-style alias
        "num_threads": _LGBM_INNER_THREADS, # canonical name used by lgb.train (some builds ignore n_jobs here)
        "verbose": -1,
    }

    horizon_cols = [c for c in feature_data.columns if c.startswith("h") and c[1:].isdigit()]
    mi_matrix = feature_data.set_index("feature")[horizon_cols]  # (n_features, n_horizons)
    available_features = [f for f in mi_matrix.index if f in features.columns]  # Only keep features that exist in the in-memory features DataFrame
    print(f"  available features: {len(available_features)} of {len(mi_matrix)} (intersection with loaded features parquet)", flush=True)

    # Time-range filter then random row subsample
    shared_idx = features.index.intersection(future_prediction_targets_agg.index)
    shared_idx = shared_idx[(shared_idx >= subsample_start) & (shared_idx <= subsample_end)]
    n_total = len(shared_idx)
    n_rows = min(subsample_amount, n_total)
    if n_rows < n_total:
        rng = np.random.default_rng(42)
        chosen = np.sort(rng.choice(n_total, size=n_rows, replace=False))
        shared_idx = shared_idx[chosen]
    print(f"CV subsample: {n_rows:,} rows (of {n_total:,} in range {subsample_start.date()} – {subsample_end.date()})", flush=True)

    # Arrays live in the main process; threads share them without copying (no pickling, no mmap needed)
    print(f"  materialising X_arr / Y_arr / mi_arr as float32 ndarrays", flush=True)
    _t = _time.perf_counter()
    # to_numpy(copy=False) avoids a redundant cast when underlying blocks are already float32.
    X_arr = features.loc[shared_idx, available_features].to_numpy(dtype=np.float32, copy=False)  # (n_rows, n_features)
    # Fortran (column-major) layout: LightGBM consumes data column-wise; F-order eliminates the
    # internal transpose that happens on every fit() call with C-order input. ~10-20% faster fits.
    if not X_arr.flags["F_CONTIGUOUS"]:
        X_arr = np.asfortranarray(X_arr)
    Y_arr = future_prediction_targets_agg.loc[shared_idx].to_numpy(dtype=np.float32, copy=False)  # (n_rows, n_horizons)
    if not Y_arr.flags["C_CONTIGUOUS"]:
        Y_arr = np.ascontiguousarray(Y_arr)
    mi_arr = mi_matrix.loc[available_features].to_numpy(dtype=np.float32, copy=False)             # (n_features, n_horizons)
    if not mi_arr.flags["C_CONTIGUOUS"]:
        mi_arr = np.ascontiguousarray(mi_arr)
    print(
        f"    arrays ready in {_time.perf_counter() - _t:.1f}s | "
        f"X={X_arr.nbytes / 1024**2:.0f}MB Y={Y_arr.nbytes / 1024**2:.0f}MB mi={mi_arr.nbytes / 1024**2:.0f}MB",
        flush=True,
    )

    x_mb = X_arr.nbytes / 1e6
    print(f"X_arr: {X_arr.shape} ({x_mb:.0f} MB) | threads: {N_PARALLEL_JOBS}", flush=True)

    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
    fold_indices = list(tscv.split(X_arr))  # pre-compute once

    # Equally-spaced grid from K_MIN → K_MAX (defaulting to 1 → n_available_features), deduplicated and sorted.
    # Bounds are clipped to [1, n_available_features] so out-of-range values just collapse to the edge.
    _k_lo = 1 if K_MIN is None else max(1, int(K_MIN))
    _k_hi = len(available_features) if K_MAX is None else min(len(available_features), int(K_MAX))
    if _k_lo > _k_hi:
        _k_lo, _k_hi = _k_hi, _k_lo
    k_grid = sorted(set(int(round(k)) for k in np.linspace(_k_lo, _k_hi, N_K_VALUES)))

    # Equally-spaced horizon indices to evaluate; None = all horizons
    n_h = len(horizon_cols)
    if N_HORIZON_STEPS is None or N_HORIZON_STEPS >= n_h:
        horizon_indices = list(range(n_h))
    else:
        horizon_indices = sorted(set(int(round(i)) for i in np.linspace(0, n_h - 1, N_HORIZON_STEPS)))

    n_tasks = len(horizon_indices)  # one task per horizon — folds × k looped inside
    # Refine outer/inner thread split now that we know the real task count
    N_PARALLEL_JOBS = max(1, min(_NCPU, n_tasks))
    _LGBM_INNER_THREADS = max(1, _NCPU // N_PARALLEL_JOBS)
    _LGBM_CV_PARAMS["n_jobs"] = _LGBM_INNER_THREADS
    _LGBM_CV_PARAMS["num_threads"] = _LGBM_INNER_THREADS
    print(
        f"Horizons: {len(horizon_indices)} of {n_h} | k grid: {k_grid} | folds: {len(fold_indices)} | "
        f"total tasks: {n_tasks} | outer threads: {N_PARALLEL_JOBS} | lgbm inner threads: {_LGBM_INNER_THREADS}",
        flush=True,
    )

    # Per-horizon worker — sorts X by MI once, then sweeps folds × k_grid in-process.
    # Optimisations vs prior layout:
    #   1. argsort + X reorder happens once per HORIZON (was: once per (h, fold)). Saves N_CV_SPLITS×
    #      sort/copy work per horizon. Critical because X_sorted is the dominant memory traffic.
    #   2. Fold-level fancy-index slicing happens once per fold (was: once per (h, fold)).
    #   3. Finite-mask computed once per fold; column slice X[:, :k] is zero-copy on F-order arrays.
    #   4. Uses lgb.train + lgb.Dataset directly (skips sklearn wrapper overhead, ~5-15% faster).
    #   5. No eval_set / no early stopping — fixed n_estimators=100 (see _LGBM_CV_PARAMS).
    # backend="loky": LightGBM's C++ Dataset/train are NOT thread-safe on Windows (concurrent calls
    # race in the Dataset allocator and segfault with "access violation in LGBM_DatasetFree").
    # Process workers each get their own C++ state. Joblib memmaps X_arr/Y_arr/mi_arr (>1MB) so
    # workers share the underlying buffers zero-copy on Linux/macOS, and copy on Windows.
    def _eval_horizon(h_idx, X_arr, Y_arr, mi_arr, fold_indices, k_grid, train_params, num_boost_round):
        import warnings
        import numpy as np
        import lightgbm as lgb
        import time as _time
        warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)
        _t_h = _time.perf_counter()
        # Lightweight liveness print so the user sees activity before the first horizon completes
        # (tqdm with return_as="generator_unordered" only ticks per finished horizon).
        print(f"    [h={h_idx}] start", flush=True)

        # Sort feature columns by MI for this horizon — done once, reused for every fold and every k.
        order_h = np.argsort(mi_arr[:, h_idx])[::-1]
        X_sorted = np.asfortranarray(X_arr[:, order_h])  # one (n_rows, n_features) copy per horizon
        y = Y_arr[:, h_idx]

        per_k_fold_maes = {k: [] for k in k_grid}

        for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
            # Slice train/val rows once — these are the expensive fancy-index copies.
            X_tr_full = X_sorted[train_idx]
            X_va_full = X_sorted[val_idx]
            y_tr, y_va = y[train_idx], y[val_idx]
            mask_tr = np.isfinite(y_tr)
            mask_va = np.isfinite(y_va)
            if mask_tr.sum() < 50 or mask_va.sum() < 10:
                continue

            X_tr_masked = X_tr_full[mask_tr]
            X_va_masked = X_va_full[mask_va]
            y_tr_masked = y_tr[mask_tr]
            y_va_masked = y_va[mask_va]

            for k in k_grid:
                # Zero-copy column views on F-order arrays.
                X_tr_k = X_tr_masked[:, :k]
                X_va_k = X_va_masked[:, :k]
                # lgb.Dataset built per (fold, k) — bin construction is unavoidable when columns change,
                # but skipping the sklearn wrapper still saves measurable overhead per fit.
                dtrain = lgb.Dataset(X_tr_k, label=y_tr_masked, free_raw_data=False)
                booster = lgb.train(train_params, dtrain, num_boost_round=num_boost_round)
                preds = booster.predict(X_va_k)
                mae = float(np.mean(np.abs(y_va_masked - preds)))
                per_k_fold_maes[k].append(mae)
                del booster, dtrain
            del X_tr_full, X_va_full, X_tr_masked, X_va_masked

        del X_sorted
        print(f"    [h={h_idx}] done in {_time.perf_counter() - _t_h:.1f}s", flush=True)
        return h_idx, per_k_fold_maes

    tasks = list(horizon_indices)

    # Strip eval_set-only params from the dict before passing to lgb.train.
    _train_params = {k: v for k, v in _LGBM_CV_PARAMS.items() if k != "n_estimators"}
    _num_boost_round = _LGBM_CV_PARAMS["n_estimators"]

    print(f"  dispatching {len(tasks)} CV tasks across {N_PARALLEL_JOBS} workers (first results may take a few seconds)", flush=True)
    _t_dispatch = _time.perf_counter()
    # backend="loky" → process workers (LightGBM C++ is not thread-safe on Windows).
    # max_nbytes="1M" → joblib memmaps any arg larger than 1MB so X_arr/Y_arr/mi_arr are shared
    # via the OS page cache rather than pickled per task.
    results_gen = Parallel(
        n_jobs=N_PARALLEL_JOBS,
        backend="loky",
        batch_size=1,
        max_nbytes="1M",
        return_as="generator_unordered",
    )(
        delayed(_eval_horizon)(h_idx, X_arr, Y_arr, mi_arr, fold_indices, k_grid, _train_params, _num_boost_round)
        for h_idx in tasks
    )

    fold_mae_acc = {}
    _first = True
    for h_idx, per_k_fold_maes in tqdm(results_gen, total=len(tasks), desc="CV feature search"):
        if _first:
            print(f"    first CV result received after {_time.perf_counter() - _t_dispatch:.1f}s", flush=True)
            _first = False
        for k, maes in per_k_fold_maes.items():
            fold_mae_acc[(h_idx, k)] = maes

    cv_records = [
        {"horizon": horizon_cols[h_idx], "k": k, "cv_mae": float(np.mean(maes)) if maes else float("inf")}
        for (h_idx, k), maes in fold_mae_acc.items()
    ]

    cv_df = pd.DataFrame(cv_records)
    best_idx = cv_df.groupby("horizon")["cv_mae"].idxmin()
    horizon_best_k = cv_df.loc[best_idx].rename(columns={"k": "best_k"}).reset_index(drop=True)

    display(horizon_best_k)
    return horizon_best_k

horizon_best_k = feature_data_selection(
    feature_data_unique,
    features,
    future_prediction_targets_agg,
    subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_START"]),
    subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_END"]),
    subsample_amount=int(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_out_path = resolve(f"3_Features_select/Selected_features/{_target}_horizon_best_k_{_stem}.parquet")
print(f"Writing horizon_best_k parquet: {_out_path}", flush=True)
_t = _time.perf_counter()
horizon_best_k.to_parquet(_out_path)
print(f"  write done in {_time.perf_counter() - _t:.1f}s", flush=True)


Loading unique feature_data parquet: \\wsl.localhost\Ubuntu\home\daniel\Projects\Forecasting\3_Features_select\..\3_Features_select\Selected_features\NSW_feature_data_unique_1_dispatch_price.parquet
  loaded feature_data_unique: shape=(401, 102) in 0.1s
Loading features parquet: \\wsl.localhost\Ubuntu\home\daniel\Projects\Forecasting\3_Features_select\..\2_Features_build\Feature_data\1_dispatch_price.parquet
  loaded features: shape=(736417, 634) in 17.4s
Loading aggregated targets parquet: \\wsl.localhost\Ubuntu\home\daniel\Projects\Forecasting\3_Features_select\..\3_Features_select\Selected_features\NSW_targets_agg_1_dispatch_price.parquet
  loaded targets_agg: shape=(736417, 96) in 5.6s
  available features: 401 of 401 (intersection with loaded features parquet)
CV subsample: 1,000 rows (of 420,769 in range 2019-01-01 – 2023-01-01)
  materialising X_arr / Y_arr / mi_arr as float32 ndarrays
    arrays ready in 0.6s | X=2MB Y=0MB mi=0MB
X_arr: (1000, 401) (2 MB) | threads: 7
Horizons:

CV feature search:   0%|          | 0/96 [00:00<?, ?it/s]

    first CV result received after 26.4s


CV feature search: 100%|██████████| 96/96 [04:32<00:00,  2.84s/it]


,horizon,best_k,cv_mae
0,h1,1,42.695696
1,h10,45,49.970172
2,h11,45,56.217006
3,h12,45,46.002751
4,h13,45,58.173749
5,h14,90,66.798711
6,h15,45,65.397382
7,h16,90,54.176585
8,h17,45,44.553406
9,h18,45,40.373227


Writing horizon_best_k parquet: \\wsl.localhost\Ubuntu\home\daniel\Projects\Forecasting\3_Features_select\..\3_Features_select\Selected_features\NSW_horizon_best_k_1_dispatch_price.parquet
  write done in 0.0s
